# Rift 02: a model file is a zip full of pickles

Last time I saw that a pickle runs code the moment you load it. But a real PyTorch file is not a bare pickle sitting on disk. When you call `torch.save`, it actually writes a ZIP archive, and the pickle lives inside it next to the raw tensor data.

That detail matters more than it looks, because it is where the whole "rift" idea comes from. The scanner and the loader can disagree about what is even inside the zip, and the attacker hides in that disagreement.

I do not have PyTorch installed here, and I do not need it. A `.pt` file is just a zip, so I can build one and take it apart with Python's own `zipfile` module. Same structure, no heavyweight dependency.

In [ ]:
import io
import zipfile
import pickle
import pickletools

## Build a fake model file

We make a small, harmless pickle. It stands in for a real model's data. Later we'll compare a dangerous pickle against this one.

In [ ]:
# stand-in for model weights
fake_weights = {"layer1.weight": [0.1, 0.2, 0.3], "layer1.bias": [0.0]}

data_pk1 = pickle.dumps(fake_weights, protocol=2)
print(len(data_pk1), "bytes")
pickletools.dis(data_pk1)

* fake_state = is a normal dictionary. Real models store their weights in a structure like this.

* pickle.dumps = turns the dictionary into pickle bytes. protocol=2 is the pickle version, and it's the one PyTorch uses.

* print(len()) = shows how many bytes we made.

* pickletools.dis() = prints the instructions inside the pickle, one per line.

In [ ]:
with zipfile.ZipFile(io.BytesIO(model_bytes)) as z:
    print("files inside:")
    for name in z.namelist():
        print("  ", name)

## Look inside the pickle, without loading it

Now I pull the pickle out of the archive and disassemble it, exactly like lesson 1, but this time it came from inside a "model file". Still no execution, just reading the instructions.

In [ ]:
with zipfile.ZipFile(io.BytesIO(model_bytes)) as z:
    inner = z.read("model/data.pkl")

pickletools.dis(inner)

All it does is rebuild the dictionary. Clean, same as a safe file in lesson 1.

## Now the rift

Here is the trick that breaks naive scanners. A lazy scanner assumes the pickle is always the one file it expects, `data.pkl`, and only checks that. But the zip can hold more than one pickle, and depending on how the loader is written, it may reach the others too.

Let me build a model file with a second, sneaky pickle hidden under a name a lazy scanner would not think to check.

In [ ]:
class Sneaky:
    def __reduce__(self):
        return (print, ("  >>> the hidden pickle ran <<<",))

buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    z.writestr("model/data.pkl", pickle.dumps(fake_weights))   # the innocent one
    z.writestr("model/extra.pkl", pickle.dumps(Sneaky()))      # the hidden one

booby_trapped = buf.getvalue()

Now a scanner that only looks at `data.pkl`. This is roughly how a too-trusting tool behaves: it checks the expected file, sees nothing scary, and gives the whole archive a clean bill of health.

In [ ]:
def naive_scan(model_bytes):
    with zipfile.ZipFile(io.BytesIO(model_bytes)) as z:
        inner = z.read("model/data.pkl")          # only ever looks here
    ops = [op.name for op, arg, pos in pickletools.genops(inner)]
    dangerous = {"REDUCE", "GLOBAL", "STACK_GLOBAL"} & set(ops)
    return "SUSPICIOUS" if dangerous else "looks clean"

print("naive scanner says:", naive_scan(booby_trapped))

It says clean. But the dangerous pickle is right there in the archive, the scanner just never looked at it. If I check every pickle instead of only the expected one, the picture changes completely.

In [ ]:
def thorough_scan(model_bytes):
    findings = []
    with zipfile.ZipFile(io.BytesIO(model_bytes)) as z:
        for name in z.namelist():
            if not name.endswith((".pkl", ".pickle")):
                continue
            ops = [op.name for op, arg, pos in pickletools.genops(z.read(name))]
            if {"REDUCE", "GLOBAL", "STACK_GLOBAL"} & set(ops):
                findings.append(name)
    return findings

hits = thorough_scan(booby_trapped)
print("thorough scanner flagged:", hits if hits else "nothing")

There it is. The gap between what the scanner bothered to check and what the loader would actually process is the rift, and it is exactly the shape of the real 2025 bypasses. Mine is the simplest possible version, a second file under a different name. The real ones use odd extensions, broken zip headers and compression tricks to widen the same gap, but the principle is identical.

One honest catch: my "thorough" scan still only trusts files ending in `.pkl` or `.pickle`. That extension check is itself a weakness, because the loader does not care what a file is called. Fixing that (deciding whether something is a pickle by its actual content, not its name) is a job for a later lesson.

So the first real task for the scanner is boring but essential: find and inspect *every* pickle in the archive, not just the one it was told to expect. That is where Layer 1 begins.

Next up: collecting the dangerous opcodes properly, and turning `thorough_scan` into the start of a real tool.